In [ ]:
import sys
!{sys.executable} -m pip install --quiet tf-keras
!{sys.executable} -m pip install --quiet nltk
!{sys.executable} -m pip install --upgrade --quiet ipywidgets
!{sys.executable} -m pip install --quiet bertopic
!{sys.executable} -m pip install --quiet ipywidgets
!{sys.executable} -m pip install --quiet nbformat
!{sys.executable} -m pip install --quiet pysentimiento
!{sys.executable} -m pip install --quiet pyarrow==21.0
!{sys.executable} -m pip install --quiet --upgrade numpy
!{sys.executable} -m pip install --quiet wordcloud


In [5]:
import json
import pandas as pd

comments = pd.read_json('../comments_info_2.json', lines=True).drop_duplicates(subset=['comment_id'], keep='first')
comments = comments[comments['comment_id'] != 'UgwQjCjoEKnEwYto6Hx4AaABAg'].reset_index(drop=True)
classif = pd.read_json('../gpt-4o_results.json', lines=True)
classif = classif.rename(columns={'id' : 'video_id'})
classif = classif[classif['video_id'] != '2V5KZJgi_WA']
merged = comments.merge(classif[['video_id', 'classification']], on='video_id', how='left')
comments_yes = merged[merged['classification'] == 'sim'].reset_index(drop=True)
comments_yes = comments_yes[comments_yes['author_channel_id'] != 'UCm3WFBDtK-hwx0rnAc0Mbew']
print(len(comments_yes))

comments_yes = comments_yes.drop_duplicates(subset=['comment'], keep='first').reset_index(drop=True)
print(len(comments_yes))


237602
222287


In [6]:
import json
import nltk
import re
from nltk.corpus import stopwords
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from hdbscan import HDBSCAN
from umap import UMAP
from IPython.display import display
import pandas as pd

def limpar_texto_func(texto, stop_words):
    if not isinstance(texto, str): return ""
    texto = texto.lower()
    # 1. Lowercase
    texto = texto.lower()
    
    # 2. Remover URLs
    texto = re.sub(r"http\S+", "", texto)
    
    # 3. **Compressão de caracteres repetidos** (kkkkkkk → kkk)
    texto = re.sub(r"(.)\1{2,}", r"\1\1\1", texto)
    
    # 4. Remover emojis e símbolos (mantém letras acentuadas)
    texto = re.sub(r"[^a-zA-ZÀ-ÿ\s]", "", texto)
    palavras = texto.split()
    palavras = [p for p in palavras if p not in stop_words and len(p) > 2]
    return " ".join(palavras)


2025-12-08 23:05:11.876175: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [9]:
import os
import json
from tqdm import tqdm
from bertopic import BERTopic
from hdbscan import HDBSCAN
from umap import UMAP
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords
import nltk

# --------------------------------
# Função original (sem plotagem)
# --------------------------------
from tqdm import tqdm

def executar_analise_bertopic(df, limpar_texto_func, params):

    etapas = [
        "Baixando stopwords",
        "Limpando textos",
        "Criando vectorizer",
        "Criando UMAP",
        "Criando HDBSCAN",
        "Executando BERTopic",
        "Gerando resultados"
    ]

    pbar = tqdm(total=len(etapas), desc="Execução do modelo", ncols=100)

    # === STOPWORDS ===
    nltk.download('stopwords', quiet=True)
    stop_words = set(stopwords.words("portuguese"))
    stop_words.update(["pra", "vc", "pq", "porque", "tbm", "sobre", "kkk"])
    pbar.update(1)

    # === LIMPEZA ===
    mensagens_limpas = []
    mensagens_originais = []

    for _, row in df.iterrows():
        try:
            conteudo = row["comment"].strip()
            if conteudo:
                txt = limpar_texto_func(conteudo, stop_words)
                if txt:
                    mensagens_limpas.append(txt)
                    mensagens_originais.append(conteudo)
        except:
            continue

    pbar.update(1)

    # === MODELOS ===
    vectorizer_model = CountVectorizer(
        ngram_range=params["ngram_range"],
        stop_words=list(stop_words)
    )
    pbar.update(1)

    umap_model = UMAP(
        n_neighbors=params["n_neighbors"],
        n_components=params["n_components"],
        min_dist=params["min_dist"],
        metric='cosine',
        random_state=42
    )
    pbar.update(1)

    hdbscan_model = HDBSCAN(
        min_cluster_size=params["min_cluster_size"],
        min_samples=params["min_samples"],
        cluster_selection_epsilon=params["cluster_selection_epsilon"],
        metric='euclidean',
        cluster_selection_method='eom'
    )
    pbar.update(1)

    # === BERTopic ===
    topic_model = BERTopic(
        language="portuguese",
        vectorizer_model=vectorizer_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        min_topic_size=params["min_topic_size"],
        verbose=True
    )

    topics, probs = topic_model.fit_transform(mensagens_limpas)
    pbar.update(1)

    df_info = topic_model.get_topic_info()
    doc_info = topic_model.get_document_info(mensagens_limpas)
    doc_info["texto_original"] = mensagens_originais
    pbar.update(1)

    pbar.close()

    return {
        "topic_model": topic_model,
        "df_info": df_info,
        "topics": topics,
        "probs": probs,
        "doc_info": doc_info,
    }



# ==========================
#     EXECUÇÃO EM LOTE
# ==========================

def rodar_experimentos(df, limpar_texto_func):

    # Diretório de saída
    OUTPUT_DIR = "resultados_bertopic_7"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Listas de parâmetros para testar
    param_grid = {
        "ngram_range": [(1, 2)],
        "n_neighbors": [15],
        "n_components": [10],
        "min_dist": [0.0],
        "min_cluster_size": [250],
        "min_samples": [15],
        "cluster_selection_epsilon": [0.75, 1.00],
        "min_topic_size": [250],
    }

    # Geração do espaço de busca (lista de dicionários)
    from itertools import product

    keys, values = zip(*param_grid.items())
    param_combinations = [dict(zip(keys, v)) for v in product(*values)]

    print(f"Total de combinações de parâmetros: {len(param_combinations)}")

    # Loop com tqdm
    for i, params in enumerate(tqdm(param_combinations, desc="Rodando experimentos")):
        
        print("\n-----------------------------------------")
        print(f"Execução {i+1}/{len(param_combinations)}")
        print("Parâmetros:", params)

        resultados = executar_analise_bertopic(df, limpar_texto_func, params)

        # Diretório da execução
        run_dir = os.path.join(OUTPUT_DIR, f"execucao_{i:03d}")
        os.makedirs(run_dir, exist_ok=True)

        # === salvar df_info ===
        resultados["df_info"].to_csv(os.path.join(run_dir, "df_info.csv"), index=False)

        # === salvar doc_info ===
        resultados["doc_info"].to_csv(os.path.join(run_dir, "doc_info.csv"), index=False)

        # === salvar parâmetros ===
        with open(os.path.join(run_dir, "parametros.json"), "w", encoding="utf-8") as f:
            json.dump(params, f, indent=4, ensure_ascii=False)

        # === salvar modelo ===
        resultados["topic_model"].save(os.path.join(run_dir, "modelo_bertopic"))

        # Log simples
        with open(os.path.join(run_dir, "log.txt"), "w", encoding="utf-8") as f:
            f.write(str(resultados["df_info"].head()))


    print("\n\n>>> EXPERIMENTOS CONCLUÍDOS COM SUCESSO! <<<")
    print(f"Resultados salvos em: {OUTPUT_DIR}")


In [10]:
rodar_experimentos(comments_yes, limpar_texto_func)

Total de combinações de parâmetros: 2


Rodando experimentos:   0%|          | 0/2 [00:00<?, ?it/s]


-----------------------------------------
Execução 1/2
Parâmetros: {'ngram_range': (1, 2), 'n_neighbors': 15, 'n_components': 10, 'min_dist': 0.0, 'min_cluster_size': 250, 'min_samples': 15, 'cluster_selection_epsilon': 0.75, 'min_topic_size': 250}



Execução do modelo:  29%|████████████▊                                | 2/7 [00:13<00:33,  6.71s/it]2025-12-09 00:22:48,749 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/6783 [00:00<?, ?it/s]

...


-----------------------------------------
Execução 2/2
Parâmetros: {'ngram_range': (1, 2), 'n_neighbors': 15, 'n_components': 10, 'min_dist': 0.0, 'min_cluster_size': 250, 'min_samples': 15, 'cluster_selection_epsilon': 1.0, 'min_topic_size': 250}



Execução do modelo:  29%|████████████▊                                | 2/7 [00:14<00:35,  7.05s/it]2025-12-09 00:48:30,496 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/6783 [00:00<?, ?it/s]

2025-12-09 01:05:36,330 - BERTopic - Embedding - Completed ✓
2025-12-09 01:05:36,332 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-12-09 01:12:46,837 - BERTopic - Dimensionality - Completed ✓
2025-12-09 01:12:46,847 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-12-09 01:13:02,899 - BERTopic - Cluster - Completed ✓
2025-12-09 01:13:02,933 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-12-09 01:13:16,345 - BERTopic - Representation - Completed ✓

Execução do modelo: 100%|████████████████████████████████████████████| 7/7 [25:00<00:00, 214.39s/it]
2025-12-09 01:13:22,624 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.
Rodando experimentos: 100%|██████████| 2/2 [51:21<00:00, 1540.98s/it]



>>> EXPERIMENTOS CONCLUÍDOS COM SUCESSO! <<<
Resultados salvos em: resultados_bertopic_7
